# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [5]:
from PIL import Image
import os
from torch.utils.data import Dataset, dataset, random_split

Set age brackets we are going to use

In [6]:
#function to get a the age bracket each image should be in
def get_bracket(age):
        if age < 9:
            return 0
        elif age < 20:
            return 1
        elif age <= 30:
            return 2
        elif age <= 40:
            return 3
        elif age <= 50:
            return 4
        elif age <= 60:
            return 5
        elif age <= 70:
            return 6
        #anybody older than 70 would be in the same group since they tend to look the same facial features
        elif age > 70:
            return 7
        else:
            #invalid images will get placed in their own list
            return -1

Load in images into dataframe

In [7]:

class AgeDataSet(Dataset):
    def __init__(self, file_path, transform=None):
        self.transform = transform
        self.samples = []

    #load images into python list, key is path value is age
        for file in os.listdir(file_path):
            try:
                age = int(file.split("_")[0])
                label = get_bracket(age)

                path = os.path.join(file_path, file)

                self.samples.append([path, label])

            except(IndexError, ValueError) as e:
                print (f"Error proccessing {file}. Error: {e}")

    #return the size of our dataset
    def __len__(self):
        return len(self.samples)

    #return an image of requested index
    def __getitem__(self, index):
        path,label = self.samples[index]

        img = Image.open(path)
        if self.transform:
            img = self.transform(img)
        return img, label

Load the data into the python list while transforming into a 224 x 224 image and nomalizing it.

In [10]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    #convert the images to grayscale to train faster over one chanel
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

dataset = AgeDataSet(file_path="./data/images", transform=transform)


Define the neural network and its layers

In [14]:
from torch.utils.data import DataLoader
import torch.nn as nn

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels)
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x              # save input
        out = self.block(x)       # pass through layers
        out = out + residual      # add input back (skip connection)
        out = self.relu(out)      # activate after addition
        return out

class NeuralNetwork(nn.Module):
    def __init__(self, num_classes=8):
        super(NeuralNetwork, self).__init__()

        self.block1 = nn.Sequential(  #covolution block 1 and 2
            nn.Conv2d(1,32,kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.res1 = ResidualBlock(32)  #this is the residual skip connection to help against gradient loss


        self.block3 = nn.Sequential(
            nn.Conv2d(64,128,kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.block4 = nn.Sequential(
            nn.Conv2d(128,256,kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.res2 = ResidualBlock(256) #second residual block

        self.block5 = nn.Sequential(
            nn.Conv2d(256,512,kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.res1(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.res2(x)
        x = self.block5(x)
        x = self.classifier(x)
        return x

Training and validation data splits

In [20]:
import torch
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))        #make it an 80/20 split for traning and validation
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

#load the data into the loader using 32 images per batch
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   #attempt to use gpu if not cpu

model = NeuralNetwork(num_classes=8).to(device) #define model

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



Training Loop

In [21]:
#train over 20 generations
num_epochs = 20

for epoch in range(num_epochs):

#train the mdoel
    train_loss = 0
    train_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    #validating the model
    model.eval()
    val_loss = 0
    val_correct = 0

    with torch.no_grad():  # no gradients needed for validation
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

#print the training stats
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {100*train_correct/len(train_dataset):.2f}%")
    print(f"  Val Loss:   {val_loss/len(val_loader):.4f} | Val Acc:   {100*val_correct/len(val_dataset):.2f}%")

# 4. save the model
torch.save(model.state_dict(), "age_model.pth")
print("Model saved.")

RuntimeError: Given groups=1, weight of size [32, 32, 3, 3], expected input[32, 64, 56, 56] to have 32 channels, but got 64 channels instead